In [3]:
import pandas as pd
import numpy as np

In [4]:
df=pd.read_csv("data/df_limpio.csv")

In [5]:
df.head()

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,Estudios,Tipo_Jornada_Laboral,Estado_Civil,Posesion_Hipoteca,Personas_Cargo,Proposito,Fiador,Impago,Prima
0,6B4H6E,22,16000.0,10000,769,22.0,1,8.96,48.0,0.32,Escolar,Desempleado,Soltero,1.0,1.0,Vivienda,0.0,0,65.85
1,VKTUXP,33,38308.0,16474,699,70.0,3,14.07,36.0,0.20,Máster,Jornada completa,Casado,1.0,1.0,Vivienda,0.0,0,260.60
2,5P9OVL,51,51271.0,40770,812,309.0,1,12.66,12.0,0.36,Grado Universitario,Jornada completa,Casado,0.0,1.0,Vivienda,1.0,0,221.02
3,YL0W8G,30,31923.0,36707,813,43.0,1,8.96,24.0,0.32,Grado Universitario,Jornada completa,Casado,0.0,1.0,Vivienda,0.0,0,125.00
4,YP8WTV,28,27373.0,86981,765,60.0,2,10.07,48.0,0.55,Escolar,Jornada completa,Soltero,0.0,1.0,Vivienda,0.0,0,800.00


In [6]:
import pandas as pd 
#Para calcular los prestamos solo necesitamos las columnas ID, Monto_Inicial, Ratio_Interes y Duracion por lo que creamos un nuevo dataframe con estas columnas.
cols_necesarias = ["ID", "Monto_Inicial", "Ratio_Interes", "Duracion"]
prestamos = df[cols_necesarias].copy()
print(prestamos)

            ID  Monto_Inicial  Ratio_Interes  Duracion
0       6B4H6E          10000           8.96      48.0
1       VKTUXP          16474          14.07      36.0
2       5P9OVL          40770          12.66      12.0
3       YL0W8G          36707           8.96      24.0
4       YP8WTV          86981          10.07      48.0
...        ...            ...            ...       ...
101890  5PM51H           5681          12.11      48.0
101891  L0RUJT          40000          10.77      48.0
101892  8OJZU0          41795          10.41      36.0
101893  II8B24          21847           6.26      36.0
101894  A8KWSO          40000           6.44      24.0

[101895 rows x 4 columns]


In [ ]:
import pandas as pd

filas = []
for _, p in prestamos.iterrows():
    capital = p["Monto_Inicial"]
    n = int(p["Duracion"])
    i = p["Ratio_Interes"] / 100 / 12  # tipo de interés mensual

    # Evitar división por cero cuando el interés es 0 %.
    cuota_fr = capital / n if i == 0 else capital * i / (1 - (1 + i) ** -n)
    amort_al = capital / n

    saldo_fr = capital
    saldo_al = capital
    for mes in range(1, n + 1):
        interes_fr = saldo_fr * i
        amort_fr = cuota_fr - interes_fr
        interes_al = saldo_al * i
        cuota_al = amort_al + interes_al

        filas.append({
            "ID": p["ID"], "Mes": mes,
            "Fr_Cuota": cuota_fr, "Fr_Intereses": interes_fr,
            "Fr_Capital_Amortizado": amort_fr, "Fr_Capital_Pendiente": saldo_fr - amort_fr,
            "Fr_Flujo_Caja": -amort_fr, "Al_Cuota": cuota_al, "Al_Intereses": interes_al,
            "Al_Capital_Amortizado": amort_al, "Al_Capital_Pendiente": saldo_al - amort_al,
            "Al_Flujo_Caja": -amort_al
        })
        saldo_fr -= amort_fr
        saldo_al -= amort_al

cuadros = pd.DataFrame(filas).round(2)
cuadros.to_csv('cuadros_amort.csv')


In [8]:
import pandas as pd

# 1) Primera fila (Mes 1) de cada cliente, con sufijo _M1
primera = (cuadros[cuadros["Mes"] == 1]
           .drop(columns="Mes")
           .set_index("ID")
           .add_suffix("_M1"))

# 2) Intereses totales de cada sistema
totales = cuadros.groupby("ID")[["Fr_Intereses", "Al_Intereses"]].sum()
totales.columns = ["Fr_Intereses_Total", "Al_Intereses_Total"]

# 3) Intereses cobrados hasta el impago y saldo pendiente en ese momento
MOMENTO_IMPAGO = 0.50   # el impago ocurre a mitad del plazo (supuesto)
cuadros["Mes_Impago"] = (cuadros.groupby("ID")["Mes"].transform("max")
                         * MOMENTO_IMPAGO).round().clip(lower=1).astype(int)

antes = (cuadros[cuadros["Mes"] <= cuadros["Mes_Impago"]]
         .groupby("ID")[["Fr_Intereses", "Al_Intereses"]].sum())
antes.columns = ["Fr_Intereses_Hasta_Impago", "Al_Intereses_Hasta_Impago"]

saldo = (cuadros[cuadros["Mes"] == cuadros["Mes_Impago"]]
         .set_index("ID")[["Fr_Capital_Pendiente", "Al_Capital_Pendiente"]])
saldo.columns = ["Fr_Saldo_Impago", "Al_Saldo_Impago"]

# 4) Unir todo con el dataset original (una fila por cliente)
tabla = (df.merge(primera, on="ID", how="left")
           .merge(totales, on="ID", how="left")
           .merge(antes, on="ID", how="left")
           .merge(saldo, on="ID", how="left"))

# 5) Carga de la primera cuota sobre ingresos
# Si los ingresos faltan o son cero, la carga no se puede calcular.
ingresos_validos = tabla["Ingresos"].where(tabla["Ingresos"] > 0)
tabla["Carga_Fr"] = tabla["Fr_Cuota_M1"] * 12 / ingresos_validos
tabla["Carga_Al"] = tabla["Al_Cuota_M1"] * 12 / ingresos_validos

tabla

,ID,Edad,Ingresos,Monto_Inicial,Scoring_Crediticio,Meses_Empleo,Num_Creditos,Ratio_Interes,Duracion,Ratio_Deuda_Ingresos,...,Al_Capital_Pendiente_M1,Al_Flujo_Caja_M1,Fr_Intereses_Total,Al_Intereses_Total,Fr_Intereses_Hasta_Impago,Al_Intereses_Hasta_Impago,Fr_Saldo_Impago,Al_Saldo_Impago,Carga_Fr,Carga_Al
0,6B4H6E,22,16000.0,10000,769,22.0,1,8.96,48.0,0.32,...,9791.67,-208.33,1935.70,1829.34,1413.02,1362.67,5445.15,5000.0,0.186495,0.212250
1,VKTUXP,33,38308.0,16474,699,70.0,3,14.07,36.0,0.20,...,16016.39,-457.61,3815.68,3573.42,2768.84,2655.92,9098.00,8237.0,0.176548,0.203854
2,5P9OVL,51,51271.0,40770,812,309.0,1,12.66,12.0,0.36,...,37372.50,-3397.50,2849.56,2795.80,2066.37,2043.09,21026.59,20385.0,0.850766,0.895856
3,YL0W8G,30,31923.0,36707,813,43.0,1,8.96,24.0,0.32,...,35177.54,-1529.46,3523.59,3426.00,2580.45,2535.24,19172.14,18353.5,0.630121,0.677959
4,YP8WTV,28,27373.0,86981,765,60.0,2,10.07,48.0,0.55,...,85168.90,-1812.10,19050.40,17882.94,13871.87,13320.96,47837.16,43490.5,0.968395,1.114392
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101890,5PM51H,29,35671.0,5681,628,27.0,2,12.11,48.0,0.10,...,5562.65,-118.35,1514.64,1404.60,1097.94,1046.28,3181.11,2840.5,0.050431,0.059100
101891,L0RUJT,28,22864.0,40000,752,9.0,2,10.77,48.0,0.55,...,39166.67,-833.33,9409.20,8795.48,6840.82,6551.74,22136.21,20000.0,0.540252,0.625786
101892,8OJZU0,35,69159.0,41795,699,65.0,2,10.41,36.0,0.29,...,40634.03,-1160.97,7045.02,6707.57,5143.79,4985.36,22518.77,20897.5,0.235400,0.264354
101893,II8B24,39,68375.0,21847,722,170.0,2,6.26,36.0,0.14,...,21240.14,-606.86,2172.33,2108.42,1597.32,1567.07,11434.65,10923.5,0.117095,0.126508


In [9]:
print(tabla[["Carga_Fr", "Carga_Al"]].describe().round(3))
print("Cabe en francés:", (tabla["Carga_Fr"] <= 0.35).mean().round(3))
print("Cabe en alemán: ", (tabla["Carga_Al"] <= 0.35).mean().round(3))

         Carga_Fr    Carga_Al
count  101895.000  101895.000
mean        0.499       0.552
std         0.555       0.595
min         0.016       0.019
25%         0.156       0.176
50%         0.320       0.359
75%         0.633       0.707
max         8.547       8.819
Cabe en francés: 0.535
Cabe en alemán:  0.49


In [10]:
import numpy as np

MAX_CARGA = 0.35

# Ahorro de intereses que supone elegir el alemán (en % del importe)
tabla["Ahorro_Al"] = tabla["Fr_Intereses_Total"] - tabla["Al_Intereses_Total"]
tabla["Ahorro_Al_pct"] = tabla["Ahorro_Al"] / tabla["Monto_Inicial"]

# Puntos de fragilidad del cliente (más puntos = más frágil)
p25_scoring = tabla["Scoring_Crediticio"].quantile(0.25)
p75_deuda = tabla["Ratio_Deuda_Ingresos"].quantile(0.75)
jornada = tabla["Tipo_Jornada_Laboral"]

tabla["Puntos_Fragilidad"] = (
    np.where(jornada == "Desempleado", 2, np.where(jornada == "Jornada completa", 0, 1))
    + (tabla["Meses_Empleo"] < 12).astype(int)                      # poca antigüedad
    + (tabla["Scoring_Crediticio"] < p25_scoring).astype(int)       # scoring bajo
    + (tabla["Ratio_Deuda_Ingresos"] > p75_deuda).astype(int)       # ya muy endeudado
    + (tabla["Personas_Cargo"] >= 2).astype(int)                    # familia a cargo
    + ((MAX_CARGA - tabla["Carga_Al"]) < 0.05).astype(int)          # justo en el límite
    - (tabla["Fiador"] == 1).astype(int)                            # con fiador
)
tabla["Fragil"] = tabla["Puntos_Fragilidad"] >= 2

# Recomendación por cliente
al_ok = tabla["Carga_Al"] <= MAX_CARGA
fr_ok = tabla["Carga_Fr"] <= MAX_CARGA
vale_la_pena_aleman = tabla["Ahorro_Al_pct"] >= 0.01   # el alemán ahorra al menos el 1 %

tabla["Recomendado"] = np.select(
    [al_ok & ~tabla["Fragil"] & vale_la_pena_aleman, fr_ok],
    ["Aleman", "Frances"],
    default="Ninguno",
)

# Ver el resultado
print(tabla["Recomendado"].value_counts(normalize=True).round(3))
tabla[["ID", "Recomendado", "Carga_Fr", "Carga_Al", "Puntos_Fragilidad", "Ahorro_Al"]].head(20)

Recomendado
Ninguno    0.465
Frances    0.354
Aleman     0.180
Name: proportion, dtype: float64


,ID,Recomendado,Carga_Fr,Carga_Al,Puntos_Fragilidad,Ahorro_Al
0,6B4H6E,Frances,0.186495,0.212250,2,106.36
1,VKTUXP,Aleman,0.176548,0.203854,0,242.26
2,5P9OVL,Ninguno,0.850766,0.895856,0,53.76
3,YL0W8G,Ninguno,0.630121,0.677959,1,97.59
4,YP8WTV,Ninguno,0.968395,1.114392,1,1167.46
5,F9YNI5,Ninguno,0.359435,0.422188,0,1265.85
6,98XK3H,Aleman,0.068160,0.080002,0,64.33
7,AKMVZ2,Ninguno,0.742180,0.830854,2,728.53
8,XWXHI1,Ninguno,2.987077,3.033281,0,11.59
9,BJ8987,Frances,0.194562,0.219533,0,348.29


In [11]:
print("Número de Ninguno:", (tabla["Recomendado"] == "Ninguno").sum())
print("Porcentaje de Ninguno:", (tabla["Recomendado"] == "Ninguno").mean().round(3))

# Número y porcentaje de los tres tipos
print(tabla["Recomendado"].value_counts())
print(tabla["Recomendado"].value_counts(normalize=True).round(3))

Número de Ninguno: 47405
Porcentaje de Ninguno: 0.465
Recomendado
Ninguno    47405
Frances    36119
Aleman     18371
Name: count, dtype: int64
Recomendado
Ninguno    0.465
Frances    0.354
Aleman     0.180
Name: proportion, dtype: float64


In [12]:
suma=0.465+0.354+0.180
print("Suma de los porcentajes:", suma)

Suma de los porcentajes: 0.9989999999999999


In [13]:
import pandas as pd

for p in [2, 3, 4]:
    fragil = tabla["Puntos_Fragilidad"] >= p
    rec = np.select([al_ok & ~fragil & vale_la_pena_aleman, fr_ok],
                    ["Aleman", "Frances"], default="Ninguno")
    print(p, pd.Series(rec).value_counts(normalize=True).round(3).to_dict())

2 {'Ninguno': 0.465, 'Frances': 0.354, 'Aleman': 0.18}
3 {'Ninguno': 0.465, 'Frances': 0.315, 'Aleman': 0.22}
4 {'Ninguno': 0.465, 'Frances': 0.306, 'Aleman': 0.229}


In [14]:
print(tabla.groupby("Fragil")["Impago"].agg(["mean", "count"]).round(3))
print(tabla.groupby("Puntos_Fragilidad")["Impago"].agg(["mean", "count"]).round(3))

         mean  count
Fragil              
False   0.108  65902
True    0.128  35993
                    mean  count
Puntos_Fragilidad              
-1                 0.101   4404
 0                 0.106  22909
 1                 0.110  38589
 2                 0.123  25705
 3                 0.141   8769
 4                 0.150   1446
 5                 0.205     73


In [15]:
tabla["Fragil"] = tabla["Puntos_Fragilidad"] >= 3
tabla["Recomendado"] = np.select(
    [al_ok & ~tabla["Fragil"] & vale_la_pena_aleman, fr_ok],
    ["Aleman", "Frances"],
    default="Ninguno",
)
print(tabla["Recomendado"].value_counts(normalize=True).round(3))

Recomendado
Ninguno    0.465
Frances    0.315
Aleman     0.220
Name: proportion, dtype: float64


In [16]:
for m in [0.35, 0.40, 0.45, 0.50, 0.60]:
    print(m, round((tabla["Carga_Fr"] > m).mean(), 3))

0.35 0.465
0.4 0.413
0.45 0.369
0.5 0.331
0.6 0.267


In [17]:
PLAZO_MAX = 120   # meses máximos que aceptas
cuota_max = MAX_CARGA * tabla["Ingresos"] / 12
i = tabla["Ratio_Interes"] / 100 / 12
C = tabla["Monto_Inicial"]
with np.errstate(divide="ignore", invalid="ignore"):
    plazo = np.where(
        i == 0,
        C / cuota_max,
        np.log(1 - C * i / cuota_max) / -np.log(1 + i)
    )
plazo_valido = (cuota_max > C * i) & (cuota_max > 0) & np.isfinite(plazo)
tabla["Plazo_Min_Fr"] = np.where(plazo_valido, np.ceil(plazo), np.nan)

ninguno = tabla["Recomendado"] == "Ninguno"
amplia = ninguno & (tabla["Plazo_Min_Fr"] <= PLAZO_MAX)
tabla["Recomendado_Final"] = np.where(amplia, "Frances (ampliar plazo)", tabla["Recomendado"])
print(tabla["Recomendado_Final"].value_counts(normalize=True).round(3))

Recomendado_Final
Frances (ampliar plazo)    0.330
Frances                    0.315
Aleman                     0.220
Ninguno                    0.135
Name: proportion, dtype: float64


In [18]:
amp = tabla["Recomendado_Final"] == "Frances (ampliar plazo)"
print((tabla.loc[amp, "Plazo_Min_Fr"] - tabla.loc[amp, "Duracion"]).describe().round(1))

count    33601.0
mean        28.5
std         23.4
min          1.0
25%          9.0
50%         22.0
75%         43.0
max        108.0
dtype: float64


In [19]:
dif = tabla["Plazo_Min_Fr"] - tabla["Duracion"]
for extra in [12, 24, 36, 48, 120]:
    ok = ninguno & (dif <= extra) & (tabla["Plazo_Min_Fr"] <= PLAZO_MAX)
    print(f"Hasta +{extra} meses -> Ninguno: {(ninguno & ~ok).mean():.3f}")

Hasta +12 meses -> Ninguno: 0.359
Hasta +24 meses -> Ninguno: 0.287
Hasta +36 meses -> Ninguno: 0.238
Hasta +48 meses -> Ninguno: 0.202
Hasta +120 meses -> Ninguno: 0.135


In [20]:
PLAZO_MAX = 120
EXTRA_MAX = 36

cuota_max = MAX_CARGA * tabla["Ingresos"] / 12
i = tabla["Ratio_Interes"] / 100 / 12
C = tabla["Monto_Inicial"]
n = tabla["Duracion"]
with np.errstate(divide="ignore", invalid="ignore"):
    plazo = np.where(
        i == 0,
        C / cuota_max,
        np.log(1 - C * i / cuota_max) / -np.log(1 + i)
    )
plazo_valido = (cuota_max > C * i) & (cuota_max > 0) & np.isfinite(plazo)
tabla["Plazo_Min_Fr"] = np.where(plazo_valido, np.ceil(plazo), np.nan)

ninguno = tabla["Recomendado"] == "Ninguno"
amplia = (ninguno & (tabla["Plazo_Min_Fr"] <= PLAZO_MAX)
          & ((tabla["Plazo_Min_Fr"] - n) <= EXTRA_MAX))
reducir = ninguno & ~amplia

# Importe máximo (francés, con el plazo original) para que la cuota quepa en MAX_CARGA
importe_max = np.where(
    i == 0,
    cuota_max * n,
    cuota_max * (1 - (1 + i) ** -n) / i
)
tabla["Importe_Max_Fr"] = np.where(
    (n > 0) & (cuota_max > 0) & np.isfinite(importe_max),
    importe_max,
    np.nan
).round(0)

tabla["Recomendado_Final"] = np.select(
    [amplia, reducir],
    ["Frances (ampliar plazo)", "Reducir importe"],
    default=tabla["Recomendado"],
)
print(tabla["Recomendado_Final"].value_counts(normalize=True).round(3))
tabla.loc[reducir, ["ID", "Monto_Inicial", "Importe_Max_Fr", "Duracion"]].head(10)

Recomendado_Final
Frances                    0.315
Reducir importe            0.238
Frances (ampliar plazo)    0.228
Aleman                     0.220
Name: proportion, dtype: float64


,ID,Monto_Inicial,Importe_Max_Fr,Duracion
4,YP8WTV,86981,31437.0,48.0
7,AKMVZ2,98937,46657.0,48.0
8,XWXHI1,117341,13749.0,12.0
10,3BGF0X,91977,12756.0,24.0
12,DFWIZR,136913,38232.0,24.0
13,57AKEJ,150000,47681.0,36.0
22,DROSEQ,237184,85256.0,60.0
23,ZJ395E,221677,24361.0,12.0
27,V2P8J1,77155,37456.0,36.0
41,36XL7F,64564,45603.0,60.0


In [ ]:
red = tabla["Recomendado_Final"] == "Reducir importe"
recorte = 1 - tabla.loc[red, "Importe_Max_Fr"] / tabla.loc[red, "Monto_Inicial"]
print(recorte.describe().round(2))      # fracción del préstamo que habría que quitar

tabla.to_csv("resultado_final.csv", index=False)

count    24207.00
mean         0.64
std          0.15
min          0.21
25%          0.53
50%          0.64
75%          0.75
max          0.96
dtype: float64


In [ ]:
RECORTE_MAX = 0.50    # máximo recorte del importe que consideras razonable

tabla["Recorte"] = 1 - tabla["Importe_Max_Fr"] / tabla["Monto_Inicial"]
mask = tabla["Recomendado_Final"] == "Reducir importe"
tabla.loc[mask & (tabla["Recorte"] > RECORTE_MAX), "Recomendado_Final"] = "Denegar / estudio especial"

print(tabla["Recomendado_Final"].value_counts(normalize=True).round(3))

Recomendado_Final
Frances                       0.315
Frances (ampliar plazo)       0.228
Aleman                        0.220
Denegar / estudio especial    0.190
Reducir importe               0.047
Name: proportion, dtype: float64


In [ ]:
columnas = ["ID", "Monto_Inicial", "Ingresos", "Duracion", "Carga_Fr", "Carga_Al",
            "Puntos_Fragilidad", "Recomendado", "Recomendado_Final",
            "Plazo_Min_Fr", "Importe_Max_Fr"]
tabla[columnas].head(30)

,ID,Monto_Inicial,Ingresos,Duracion,Carga_Fr,Carga_Al,Puntos_Fragilidad,Recomendado,Recomendado_Final,Plazo_Min_Fr,Importe_Max_Fr
0,6B4H6E,10000,16000.0,48.0,0.186495,0.212250,2,Aleman,Aleman,24.0,18767.0
1,VKTUXP,16474,38308.0,36.0,0.176548,0.203854,0,Aleman,Aleman,17.0,32659.0
2,5P9OVL,40770,51271.0,12.0,0.850766,0.895856,0,Ninguno,Frances (ampliar plazo),33.0,16773.0
3,YL0W8G,36707,31923.0,24.0,0.630121,0.677959,1,Ninguno,Frances (ampliar plazo),47.0,20389.0
4,YP8WTV,86981,27373.0,48.0,0.968395,1.114392,1,Ninguno,Denegar / estudio especial,294.0,31437.0
5,F9YNI5,63406,44800.0,60.0,0.359435,0.422188,0,Ninguno,Frances (ampliar plazo),63.0,61742.0
6,98XK3H,3146,19639.0,36.0,0.068160,0.080002,0,Aleman,Aleman,6.0,16154.0
7,AKMVZ2,98937,38641.0,48.0,0.742180,0.830854,2,Ninguno,Denegar / estudio especial,128.0,46657.0
8,XWXHI1,117341,40023.0,12.0,2.987077,3.033281,0,Ninguno,Denegar / estudio especial,119.0,13749.0
9,BJ8987,40000,48248.0,60.0,0.194562,0.219533,0,Frances,Frances,31.0,71956.0


In [ ]:
criterios = {
    "Desempleado": tabla["Tipo_Jornada_Laboral"] == "Desempleado",
    "Poca antigüedad (<12 meses)": tabla["Meses_Empleo"] < 12,
    "Scoring bajo (p25)": tabla["Scoring_Crediticio"] < tabla["Scoring_Crediticio"].quantile(0.25),
    "Deuda alta (p75)": tabla["Ratio_Deuda_Ingresos"] > tabla["Ratio_Deuda_Ingresos"].quantile(0.75),
    "2+ personas a cargo": tabla["Personas_Cargo"] >= 2,
    "Cerca del límite de carga": (MAX_CARGA - tabla["Carga_Al"]) < 0.05,
    "Con fiador": tabla["Fiador"] == 1,
}
for nombre, cond in criterios.items():
    print(f"{nombre:28s} {cond.mean():.3f} de clientes | impago {tabla.loc[cond,'Impago'].mean():.3f} vs {tabla.loc[~cond,'Impago'].mean():.3f}")

Desempleado                  0.099 de clientes | impago 0.118 vs 0.115
Poca antigüedad (<12 meses)  0.090 de clientes | impago 0.211 vs 0.106
Scoring bajo (p25)           0.250 de clientes | impago 0.128 vs 0.111
Deuda alta (p75)             0.000 de clientes | impago nan vs 0.115
2+ personas a cargo          0.000 de clientes | impago nan vs 0.115
Cerca del límite de carga    0.568 de clientes | impago 0.114 vs 0.117
Con fiador                   0.299 de clientes | impago 0.116 vs 0.115


In [ ]:
p25_scoring = tabla["Scoring_Crediticio"].quantile(0.25)
p75_deuda = tabla["Ratio_Deuda_Ingresos"].quantile(0.75)

tabla["Puntos_Fragilidad"] = (
    2 * (tabla["Meses_Empleo"] < 12).astype(int)                 # poca antigüedad: el criterio más fuerte
    + (tabla["Scoring_Crediticio"] < p25_scoring).astype(int)    # scoring bajo
    + (tabla["Ratio_Deuda_Ingresos"] >= p75_deuda).astype(int)   # deuda alta (con >=)
)
tabla["Fragil"] = tabla["Puntos_Fragilidad"] >= 2

tabla["Recomendado"] = np.select(
    [al_ok & ~tabla["Fragil"] & vale_la_pena_aleman, fr_ok],
    ["Aleman", "Frances"],
    default="Ninguno",
)

print(tabla.groupby("Puntos_Fragilidad")["Impago"].agg(["mean", "count"]).round(3))
print(tabla["Recomendado"].value_counts(normalize=True).round(3))

# Como han cambiado los criterios de fragilidad, recalcular también las
# recomendaciones finales para que no queden valores de las celdas anteriores.
PLAZO_MAX = 120
EXTRA_MAX = 36
RECORTE_MAX = 0.50
cuota_max = MAX_CARGA * tabla["Ingresos"] / 12
i = tabla["Ratio_Interes"] / 100 / 12
C = tabla["Monto_Inicial"]
n = tabla["Duracion"]
with np.errstate(divide="ignore", invalid="ignore"):
    plazo = np.where(
        i == 0,
        C / cuota_max,
        np.log(1 - C * i / cuota_max) / -np.log(1 + i)
    )
plazo_valido = (cuota_max > C * i) & (cuota_max > 0) & np.isfinite(plazo)
tabla["Plazo_Min_Fr"] = np.where(plazo_valido, np.ceil(plazo), np.nan)

ninguno = tabla["Recomendado"] == "Ninguno"
amplia = (ninguno & (tabla["Plazo_Min_Fr"] <= PLAZO_MAX)
          & ((tabla["Plazo_Min_Fr"] - n) <= EXTRA_MAX))
importe_max = np.where(
    i == 0,
    cuota_max * n,
    cuota_max * (1 - (1 + i) ** -n) / i
)
tabla["Importe_Max_Fr"] = np.where(
    (n > 0) & (cuota_max > 0) & np.isfinite(importe_max),
    importe_max,
    np.nan
).round(0)
reducir = ninguno & ~amplia
tabla["Recomendado_Final"] = np.select(
    [amplia, reducir],
    ["Frances (ampliar plazo)", "Reducir importe"],
    default=tabla["Recomendado"]
)
tabla["Recorte"] = 1 - tabla["Importe_Max_Fr"] / tabla["Monto_Inicial"]
mask_denegar = (tabla["Recomendado_Final"] == "Reducir importe") & (tabla["Recorte"] > RECORTE_MAX)
tabla.loc[mask_denegar, "Recomendado_Final"] = "Denegar / estudio especial"


In [ ]:
print("Cabe en alemán pero ahorra <1 %:", ((tabla["Carga_Al"] <= MAX_CARGA) & (tabla["Ahorro_Al_pct"] < 0.01)).mean().round(3))
print("Cabe en alemán, no frágil y ahorra >=1 %:", ((tabla["Carga_Al"] <= MAX_CARGA) & ~tabla["Fragil"] & (tabla["Ahorro_Al_pct"] >= 0.01)).mean().round(3))

Cabe en alemán pero ahorra <1 %: 0.26
Cabe en alemán, no frágil y ahorra >=1 %: 0.205


In [ ]:
print(tabla["Recomendado_Final"].value_counts(normalize=True).round(3))

# Exportar al final, después de actualizar todas las recomendaciones.
tabla.to_csv("resultado_final.csv", index=False)
print("Archivo guardado: resultado_final.csv")


In [ ]:
for a in [0, 0.0025, 0.005, 0.01, 0.02]:
    ok = tabla["Ahorro_Al_pct"] >= a
    rec = np.select([al_ok & ~tabla["Fragil"] & ok, fr_ok],
                    ["Aleman", "Frances"], default="Ninguno")
    print(f"Ahorro mínimo {a:.4f}:", pd.Series(rec).value_counts(normalize=True).round(3).to_dict())

Ahorro mínimo 0.0000: {'Ninguno': 0.465, 'Aleman': 0.435, 'Frances': 0.099}
Ahorro mínimo 0.0025: {'Ninguno': 0.465, 'Aleman': 0.372, 'Frances': 0.162}
Ahorro mínimo 0.0050: {'Ninguno': 0.465, 'Aleman': 0.307, 'Frances': 0.228}
Ahorro mínimo 0.0100: {'Ninguno': 0.465, 'Frances': 0.33, 'Aleman': 0.205}
Ahorro mínimo 0.0200: {'Ninguno': 0.465, 'Frances': 0.444, 'Aleman': 0.09}
